In [701]:
import pandas as pd
import numpy as np
from pathlib import Path
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', None)

In [702]:
base_path = r'C:\Users\nickc\OneDrive\Documents\Projects\Sports Performance Analytics\Section 2 - Moneyball and Beyond\Data'

In [703]:
salaries  = pd.read_csv(base_path+'/salaries.csv')

In [704]:
Master = salaries.copy()

Load the batting data and sum data across stints

In [705]:
batting = pd.read_csv(base_path+'/Batting.csv')

In [706]:
batting = batting.groupby(['playerID', 'yearID']).sum(numeric_only=True).reset_index()

Subset batting data to only include batting seasons (yearID) 1998-2006 and players with at least 130AB.

In [707]:
batting = batting[
    (batting['yearID'] >= 1998) &
    (batting['yearID'] <= 2006) &
    (batting['AB'] >= 130)
]

Calculate PA, OBP, SLG, and batting average

In [708]:
"""
SLG is defined as (Singles + 2x Doubles + 3x Triples + 4 x Home runs) / At bats
OBP is defined as (Hits + Bases on Balls + Hit by Pitch)/(At bats + Bases on Balls + Hit by Pitch + Sacrifice Flies)

Singles and Plate Appearances are not listed separately in the Lahman batting dataframe, 
but singles can be defined as Hits - Home runs - Triples - Doubles. 
Plate appearances can be defined as At bats + Base on Balls (Walk) + Hit by pitch + Sacrfice hits + Sacrifice flies.

The variables we need have mostly intuitive names: AB, H, 2B, 3B, HR, R, BB, HBP, SF, SH.
"""

'\nSLG is defined as (Singles + 2x Doubles + 3x Triples + 4 x Home runs) / At bats\nOBP is defined as (Hits + Bases on Balls + Hit by Pitch)/(At bats + Bases on Balls + Hit by Pitch + Sacrifice Flies)\n\nSingles and Plate Appearances are not listed separately in the Lahman batting dataframe, \nbut singles can be defined as Hits - Home runs - Triples - Doubles. \nPlate appearances can be defined as At bats + Base on Balls (Walk) + Hit by pitch + Sacrfice hits + Sacrifice flies.\n\nThe variables we need have mostly intuitive names: AB, H, 2B, 3B, HR, R, BB, HBP, SF, SH.\n'

In [709]:
[
    'stint',      # Sequence of teams played for within a season
    'teamID',     # Team abbreviation
    'lgID',       # League (AL or NL)
    'G',          # Games played
    'AB',         # At Bats
    'R',          # Runs
    'H',          # Hits
    'Doubles',    # 2B
    'Triples',    # 3B
    'HR',         # Home Runs
    'RBI',        # Runs Batted In
    'SB',         # Stolen Bases
    'CS',         # Caught Stealing
    'BB',         # Base on Balls (Walks)
    'SO',         # Strikeouts
    'IBB',        # Intentional Walks
    'HBP',        # Hit By Pitch
    'SH',         # Sacrifice Hits (Bunts)
    'SF',         # Sacrifice Flies
    'GIDP'        # Grounded Into Double Play
]

['stint',
 'teamID',
 'lgID',
 'G',
 'AB',
 'R',
 'H',
 'Doubles',
 'Triples',
 'HR',
 'RBI',
 'SB',
 'CS',
 'BB',
 'SO',
 'IBB',
 'HBP',
 'SH',
 'SF',
 'GIDP']

In [710]:
batting['OBP'] = (
    (batting['H'] + batting['BB'] + batting['HBP']) /
    (batting['AB'] + batting['BB'] + batting['HBP'] + batting['SF']))

batting['TB'] = (
    (batting['H'] - batting['Doubles'] - batting['Triples'] - batting['HR']) + 
    (2 * batting['Doubles']) + 
    (3 * batting['Triples']) + 
    (4 * batting['HR']))
batting['SLG'] = batting['TB'] / batting['AB']

batting['PA'] = batting['AB'] + batting['BB'] + batting['HBP'] + batting['SH'] + batting['SF']

In [711]:
batting

,playerID,yearID,stint,G,AB,R,H,Doubles,Triples,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,OBP,TB,SLG,PA
98,abbotje01,1998,1,89,244,33,68,14,1,12,41.00,3.00,3.00,9,28.00,1.00,0.00,2.00,5.00,2.00,0.30,120,0.49,260.00
100,abbotje01,2000,1,80,215,31,59,15,1,3,29.00,2.00,1.00,21,38.00,1.00,2.00,2.00,1.00,2.00,0.34,85,0.40,241.00
117,abbotku01,1998,3,77,194,26,51,13,1,5,24.00,2.00,1.00,12,53.00,0.00,2.00,1.00,3.00,5.00,0.31,81,0.42,212.00
118,abbotku01,1999,1,96,286,41,78,17,2,8,41.00,3.00,2.00,16,69.00,0.00,0.00,2.00,1.00,4.00,0.31,123,0.43,305.00
119,abbotku01,2000,1,79,157,22,34,7,1,6,12.00,1.00,1.00,14,51.00,2.00,1.00,0.00,1.00,2.00,0.28,61,0.39,173.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94988,zeileto01,2002,1,144,506,61,138,23,0,18,87.00,1.00,1.00,66,92.00,3.00,1.00,0.00,7.00,27.00,0.35,215,0.42,580.00
94989,zeileto01,2003,3,100,299,40,68,10,2,11,42.00,1.00,0.00,34,54.00,0.00,3.00,0.00,5.00,6.00,0.31,115,0.38,341.00
94990,zeileto01,2004,1,137,348,30,81,16,0,9,35.00,0.00,0.00,44,83.00,1.00,1.00,1.00,2.00,13.00,0.32,124,0.36,396.00
95105,zimmery01,2006,1,157,614,84,176,47,3,20,110.00,11.00,8.00,61,120.00,7.00,2.00,1.00,4.00,15.00,0.35,289,0.47,682.00


Create SalYear variable to create one year lag between batting performance and salary 

In [712]:
batting['SalYear'] = batting['yearID'] + 1

In [713]:
sal_agg = salaries.groupby(['playerID', 'yearID']).agg({'salary': 'sum', 'teamID': 'first'}).reset_index()
sal_agg = sal_agg.rename(columns={'yearID': 'SalYear'})

Master = pd.merge(batting, sal_agg, on=['playerID', 'SalYear'])

In [714]:
Master

,playerID,yearID,stint,G,AB,R,H,Doubles,Triples,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,OBP,TB,SLG,PA,SalYear,salary,teamID
0,abbotje01,1998,1,89,244,33,68,14,1,12,41.00,3.00,3.00,9,28.00,1.00,0.00,2.00,5.00,2.00,0.30,120,0.49,260.00,1999,255000,CHA
1,abbotje01,2000,1,80,215,31,59,15,1,3,29.00,2.00,1.00,21,38.00,1.00,2.00,2.00,1.00,2.00,0.34,85,0.40,241.00,2001,300000,FLO
2,abbotku01,1998,3,77,194,26,51,13,1,5,24.00,2.00,1.00,12,53.00,0.00,2.00,1.00,3.00,5.00,0.31,81,0.42,212.00,1999,900000,COL
3,abbotku01,1999,1,96,286,41,78,17,2,8,41.00,3.00,2.00,16,69.00,0.00,0.00,2.00,1.00,4.00,0.31,123,0.43,305.00,2000,500000,NYN
4,abbotku01,2000,1,79,157,22,34,7,1,6,12.00,1.00,1.00,14,51.00,2.00,1.00,0.00,1.00,2.00,0.28,61,0.39,173.00,2001,600000,ATL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3106,zeileto01,2000,1,153,544,67,146,36,3,22,79.00,3.00,4.00,74,85.00,4.00,2.00,0.00,3.00,15.00,0.36,254,0.47,623.00,2001,6833333,NYN
3107,zeileto01,2001,1,151,531,66,141,25,1,10,62.00,1.00,0.00,73,102.00,3.00,6.00,0.00,2.00,15.00,0.36,198,0.37,612.00,2002,6833333,COL
3108,zeileto01,2002,1,144,506,61,138,23,0,18,87.00,1.00,1.00,66,92.00,3.00,1.00,0.00,7.00,27.00,0.35,215,0.42,580.00,2003,1500000,NYA
3109,zeileto01,2003,3,100,299,40,68,10,2,11,42.00,1.00,0.00,34,54.00,0.00,3.00,0.00,5.00,6.00,0.31,115,0.38,341.00,2004,1000000,NYN


In [715]:
### --- QUIZ --- ###

Q1: What was the average player salary in 1999?  What was the average player salary in 2006?

In [716]:
avg_sals = pd.DataFrame(Master.groupby(['SalYear'])['salary'].mean()).reset_index()

print(f"Average 1999 Salary: {avg_sals.loc[avg_sals['SalYear'] == 1999, 'salary'].values[0]:.2f}")
print(f"Average 2006 Salary: {avg_sals.loc[avg_sals['SalYear'] == 2006, 'salary'].values[0]:.2f}")

Average 1999 Salary: 2242929.54
Average 2006 Salary: 3689304.76


Q2: \
Calculate the average player OBP and SLG for every season in the timeframe. \
Which season had the highest average player OBP and what was its value?  \
Which season had the highest average player SLG and what was its value? 

In [717]:
OBP_SLG = pd.DataFrame()
obp_array = Master.groupby(['yearID'])['OBP'].mean().reset_index()
slg_array = Master.groupby(['yearID'])['SLG'].mean().reset_index()
OBP_SLG = pd.merge(obp_array, slg_array, on=['yearID'])

highest_avg_obp = OBP_SLG['OBP'].max()
highest_avg_obp_year = OBP_SLG.loc[OBP_SLG['OBP'] == highest_avg_obp, 'yearID'].values[0]
highest_avg_slg = OBP_SLG['SLG'].max()
highest_avg_slg_year = OBP_SLG.loc[OBP_SLG['SLG'] == highest_avg_slg, 'yearID'].values[0]

# Using 3 decimal places as is tradition in baseball (e.g., .400 OBP)
print(f"Season with Highest Player Average OBP: {highest_avg_obp_year}, Value: {highest_avg_obp:.3f}\n"
      f"Season with Highest Player Average SLG: {highest_avg_slg_year}, Value: {highest_avg_slg:.3f}")

Season with Highest Player Average OBP: 1999, Value: 0.349
Season with Highest Player Average SLG: 2000, Value: 0.444


Q3: \
Sum HR by player across the entire timeframe.  What was the highest aggregate home run total over the timeframe 1998-2006?      

In [718]:
hr_df = Master.groupby('playerID')['HR'].sum()
hr_df.sort_values(ascending=False)

playerID
rodrial01    400
sosasa01     367
ramirma02    361
bondsba01    355
delgaca01    340
            ... 
strando01      0
ledesaa01      0
tynerja01      0
lunarfe01      0
loganno01      0
Name: HR, Length: 786, dtype: int64

Read in “People” data and extract the player’s debut year

In [719]:
people = pd.read_csv(base_path+'/People.csv')
debuts = people[['playerID', 'debut']]

Merge debut year into master data and calculate years of experience

In [720]:
Master = pd.merge(Master, debuts, on='playerID')
Master['debut'] = pd.to_datetime(Master['debut'])
Master['debut_year'] = Master['debut'].dt.year
Master['years_xp'] = Master['yearID'] - Master['debut_year']

Based on a player’s years of experience, create indicator variables for arbitration eligible players (3-6 years) and free agent players (more than 6 years) 

In [721]:
Master['arb'] = ((Master['years_xp'] >= 3) & (Master['years_xp'] <= 6))
Master['free_agent'] = (Master['years_xp'] > 6)

Read in the data for player appearances and group by stint.  Then identify the maximum number of games played at a given position for each year. 

In [722]:
appearances = pd.read_csv(base_path + '/Appearances.csv')
appearances = appearances.groupby(['yearID', 'playerID']).sum(numeric_only=True).reset_index()
appearances['max_g'] = appearances[['G_p','G_c', 'G_1b', 'G_2b', 'G_3b', 'G_ss', 'G_of','G_dh']].max(axis=1)

Create a function to determine player position.

In [723]:
appearances

,yearID,playerID,G_all,GS,G_batting,G_defense,G_p,G_c,G_1b,G_2b,G_3b,G_ss,G_lf,G_cf,G_rf,G_of,G_dh,G_ph,G_pr,max_g
0,1871,abercda01,1,1.00,1,1.00,0,0,0,0,0,1,0,0,0,0,0.00,0.00,0.00,1.00
1,1871,addybo01,25,25.00,25,25.00,0,0,0,22,0,3,0,0,0,0,0.00,0.00,0.00,22.00
2,1871,allisar01,29,29.00,29,29.00,0,0,0,2,0,0,0,29,0,29,0.00,0.00,0.00,29.00
3,1871,allisdo01,27,27.00,27,27.00,0,27,0,0,0,0,0,0,0,0,0.00,0.00,0.00,27.00
4,1871,ansonca01,25,25.00,25,25.00,0,5,1,2,20,0,1,0,0,1,0.00,0.00,0.00,20.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96607,2017,zimmejo02,29,29.00,3,29.00,29,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,29.00
96608,2017,zimmery01,144,132.00,144,143.00,0,0,143,0,0,0,0,0,0,0,1.00,4.00,0.00,143.00
96609,2017,zobribe01,128,105.00,128,118.00,0,0,5,81,0,5,36,0,32,61,0.00,20.00,0.00,81.00
96610,2017,zuninmi01,124,112.00,124,120.00,0,120,0,0,0,0,0,0,0,0,2.00,4.00,1.00,120.00


In [724]:
def player_position(df):
    if df['max_g'] == df['G_p']: return "P"
    elif df['max_g'] == df['G_c']: return "C"
    elif df['max_g'] == df['G_1b']: return "1B"
    elif df['max_g'] == df['G_2b']: return "2B"
    elif df['max_g'] == df['G_3b']: return "3B"
    elif df['max_g'] == df['G_ss']: return "SS"
    elif df['max_g'] == df['G_of']: return "OF"
    elif df['max_g'] == df['G_dh']: return "DH"
    else: return "Other"

appearances['pos'] = appearances.apply(player_position, axis=1)

Exclude non-position players.

In [725]:
appearances = appearances[appearances['max_g'] > 0]
appearances = appearances[appearances['pos'] != 'P']

Create an indicator variable for catcher and the infield (2B, SS, 3B) positions separately. Thus, you should have a separate indicator variable for 2B, SS, and 3B individually as opposed to one infielder indicator variable combining these positions. 

In [726]:
appearances['pos'].unique()

array(['SS', '2B', 'OF', 'C', '3B', '1B', 'DH'], dtype=object)

In [727]:
appearances['catcher'] = np.where(appearances['pos'] == 'C', 1, 0)

appearances['2B'] = np.where(appearances['pos'] == '2B', 1, 0)
appearances['SS'] = np.where(appearances['pos'] == 'SS', 1, 0)
appearances['3B'] = np.where(appearances['pos'] == '3B', 1, 0)

appearances['Infld'] = np.where((appearances['pos'] == "2B") | (appearances['pos'] == "3B") | 
                                (appearances['pos'] == "SS"), 1, 0)

Merge this into your master data

In [728]:
Master = pd.merge(Master, appearances, on=['yearID', 'playerID'], how='left')

In [729]:
Master

,playerID,yearID,stint,G,AB,R,H,Doubles,Triples,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,OBP,TB,SLG,PA,SalYear,salary,teamID,debut,debut_year,years_xp,arb,free_agent,G_all,GS,G_batting,G_defense,G_p,G_c,G_1b,G_2b,G_3b,G_ss,G_lf,G_cf,G_rf,G_of,G_dh,G_ph,G_pr,max_g,pos,catcher,2B,SS,3B,Infld
0,abbotje01,1998,1,89,244,33,68,14,1,12,41.00,3.00,3.00,9,28.00,1.00,0.00,2.00,5.00,2.00,0.30,120,0.49,260.00,1999,255000,CHA,1997-06-10,1997,1,False,False,89,61.00,89,76.00,0,0,0,0,0,0,20,38,27,76,2.00,15.00,0.00,76.00,OF,0,0,0,0,0
1,abbotje01,2000,1,80,215,31,59,15,1,3,29.00,2.00,1.00,21,38.00,1.00,2.00,2.00,1.00,2.00,0.34,85,0.40,241.00,2001,300000,FLO,1997-06-10,1997,3,True,False,80,52.00,80,65.00,0,0,0,0,0,0,20,33,16,65,7.00,18.00,2.00,65.00,OF,0,0,0,0,0
2,abbotku01,1998,3,77,194,26,51,13,1,5,24.00,2.00,1.00,12,53.00,0.00,2.00,1.00,3.00,5.00,0.31,81,0.42,212.00,1999,900000,COL,1993-09-07,1993,5,True,False,77,47.00,77,57.00,0,0,0,7,4,35,9,0,6,14,4.00,20.00,2.00,35.00,SS,0,0,1,0,1
3,abbotku01,1999,1,96,286,41,78,17,2,8,41.00,3.00,2.00,16,69.00,0.00,0.00,2.00,1.00,4.00,0.31,123,0.43,305.00,2000,500000,NYN,1993-09-07,1993,6,True,False,96,69.00,96,80.00,0,0,8,66,0,3,0,2,2,4,0.00,16.00,2.00,66.00,2B,0,1,0,0,1
4,abbotku01,2000,1,79,157,22,34,7,1,6,12.00,1.00,1.00,14,51.00,2.00,1.00,0.00,1.00,2.00,0.28,61,0.39,173.00,2001,600000,ATL,1993-09-07,1993,7,False,True,79,35.00,79,61.00,0,0,0,23,2,39,0,2,0,2,0.00,17.00,9.00,39.00,SS,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3106,zeileto01,2000,1,153,544,67,146,36,3,22,79.00,3.00,4.00,74,85.00,4.00,2.00,0.00,3.00,15.00,0.36,254,0.47,623.00,2001,6833333,NYN,1989-08-18,1989,11,False,True,153,148.00,153,151.00,0,0,151,0,0,0,0,0,0,0,0.00,3.00,0.00,151.00,1B,0,0,0,0,0
3107,zeileto01,2001,1,151,531,66,141,25,1,10,62.00,1.00,0.00,73,102.00,3.00,6.00,0.00,2.00,15.00,0.36,198,0.37,612.00,2002,6833333,COL,1989-08-18,1989,12,False,True,151,145.00,151,149.00,0,0,149,0,0,0,0,0,0,0,0.00,5.00,0.00,149.00,1B,0,0,0,0,0
3108,zeileto01,2002,1,144,506,61,138,23,0,18,87.00,1.00,1.00,66,92.00,3.00,1.00,0.00,7.00,27.00,0.35,215,0.42,580.00,2003,1500000,NYA,1989-08-18,1989,13,False,True,144,138.00,144,140.00,1,0,0,0,139,0,0,0,0,0,0.00,4.00,0.00,139.00,3B,0,0,0,1,1
3109,zeileto01,2003,3,100,299,40,68,10,2,11,42.00,1.00,0.00,34,54.00,0.00,3.00,0.00,5.00,6.00,0.31,115,0.38,341.00,2004,1000000,NYN,1989-08-18,1989,14,False,True,100,82.00,100,85.00,0,0,23,0,64,0,0,0,0,0,8.00,12.00,1.00,64.00,3B,0,0,0,1,1


In [730]:
### --- QUIZ 2 --- ###

What was the highest paid position on average in 1999? \
What was the highest paid position on average in 2004?

In [731]:
highest_sals = Master.groupby(['SalYear', 'pos'])['salary'].mean().reset_index()
highest_sals_99 = highest_sals[highest_sals['SalYear'] == 1999].sort_values(by='salary', ascending=False).head(1)
highest_sals_04 = highest_sals[highest_sals['SalYear'] == 2004].sort_values(by='salary', ascending=False).head(1)
high_paid_pos = pd.concat([highest_sals_99, highest_sals_04], axis=0, ignore_index=True)
high_paid_pos

,SalYear,pos,salary
0,1999,DH,3214642.86
1,2004,1B,4211004.41


What percentage of observations in the data set are either flagged as arbitration eligible or free agent eligible?

In [732]:
flagged = ((Master['arb'].sum() + Master['free_agent'].sum()) / len(Master)) * 100
print(f'{flagged:.2f}%')

78.78%


Sum years of experience by team for 2002. What is the highest and lowest aggregate years of experience for teams in 2002 data?    

In [733]:
xp02 = Master[Master['yearID'] == 2002]
xp02_agg = xp02.groupby('teamID')['years_xp'].sum().reset_index()
xp02_agg = xp02_agg.sort_values(by='years_xp').reset_index(drop=True)
highest_xp = xp02_agg.head(1)
lowest_xp = xp02_agg.tail(1)

xp = pd.concat([highest_xp, lowest_xp], ignore_index=True)
print(xp)

  teamID  years_xp
0    TBA        36
1    SFN       114


Run the following regression models:

lnSal on OBP, SLG, batting average, plate appearances, arbitration (dummy), free agent (dummy), and all positional dummy variables during the seasons prior to the publication of Moneyball (1999-2003) combined. 

In [734]:
Master['lnSal'] = np.log(Master['salary'])

In [735]:
Master['BA'] = Master['H'] / Master['AB']

In [736]:
PriorMoneyball = Master[
    (Master['SalYear'] >= 1999) &
    (Master['SalYear'] <= 2003)]

In [737]:
import statsmodels.formula.api as smf

reg1 = smf.ols(formula= 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=PriorMoneyball).fit()
reg1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  lnSal   R-squared:                       0.693
Model:                            OLS   Adj. R-squared:                  0.692
Method:                 Least Squares   F-statistic:                     491.7
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        13:09:23   Log-Likelihood:                -1845.5
No. Observations:                1750   AIC:                             3709.
Df Residuals:                    1741   BIC:                             3758.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.3703      0.164     63.357      0.000      10.049      10.691
arb[T.True]            1.2505      0.046     27.057      0.000       1.160       1.341
free_agent[T.True]     1.8274      0.046     39.513      0.000       1.737       1.918
OBP                    1.4471      0.698      2.073      0.038       0.078       2.816
SLG                    2.9605      0.312      9.487      0.000       2.348       3.573
BA                    -2.1880      0.882     -2.482      0.013      -3.917      -0.459
PA                     0.0031      0.000     27.459      0.000       0.003       0.003
catcher                0.1255      0.053      2.347      0.019       0.021       0.230
Infld                  0.0062      0.039      0.161      0.872      -0.070       0.082
==============================================================================
Omnibus:                       10.144   Durbin-Watson:                   1.390
Prob(Omnibus):                  0.006   Jarque-Bera (JB):               13.846
Skew:                           0.002   Prob(JB):                     0.000985
Kurtosis:                       3.436   Cond. No.                     2.90e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.9e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Repeat step 2) but run the regression for the seasons 2004-2006 (all years combined)

In [738]:
Moneyball = Master[
    (Master['SalYear'] >= 2004) &
    (Master['SalYear'] <= 2006)]

reg2 = smf.ols(formula= 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=Moneyball).fit()
reg2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  lnSal   R-squared:                       0.624
Model:                            OLS   Adj. R-squared:                  0.621
Method:                 Least Squares   F-statistic:                     209.2
Date:                Sun, 19 Apr 2026   Prob (F-statistic):          3.03e-208
Time:                        13:09:23   Log-Likelihood:                -1175.3
No. Observations:                1018   AIC:                             2369.
Df Residuals:                    1009   BIC:                             2413.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.0510      0.262     38.349      0.000       9.537      10.565
arb[T.True]            1.0976      0.068     16.130      0.000       0.964       1.231
free_agent[T.True]     1.6842      0.066     25.648      0.000       1.555       1.813
OBP                    5.4004      1.079      5.005      0.000       3.283       7.518
SLG                    2.9324      0.473      6.195      0.000       2.004       3.861
BA                    -4.7271      1.331     -3.552      0.000      -7.339      -2.116
PA                     0.0030      0.000     17.650      0.000       0.003       0.003
catcher                0.0947      0.078      1.218      0.224      -0.058       0.247
Infld                 -0.0277      0.056     -0.492      0.623      -0.138       0.083
==============================================================================
Omnibus:                       12.279   Durbin-Watson:                   1.388
Prob(Omnibus):                  0.002   Jarque-Bera (JB):               15.726
Skew:                           0.151   Prob(JB):                     0.000385
Kurtosis:                       3.529   Cond. No.                     3.07e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.07e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Run for each Season.

In [739]:
MB_Data_2000 = Master[(Master.SalYear == 2000)]
Val_2000_lm = smf.ols(formula = 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=MB_Data_2000).fit()
MB_Data_2001 = Master[(Master.SalYear == 2001)]
Val_2001_lm = smf.ols(formula = 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=MB_Data_2001).fit()
MB_Data_2002 = Master[(Master.SalYear == 2002)]
Val_2002_lm = smf.ols(formula = 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=MB_Data_2002).fit()
MB_Data_2003 = Master[(Master.SalYear == 2003)]
Val_2003_lm = smf.ols(formula = 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=MB_Data_2003).fit()
MB_Data_2004 = Master[(Master.SalYear == 2004)]
Val_2004_lm = smf.ols(formula = 'lnSal ~ OBP + SLG + BA + PA + arb + free_agent + catcher + Infld', data=MB_Data_2004).fit()

In [740]:
from statsmodels.iolib.summary2 import summary_col

Header = ['PriorMoneyball','Moneyball','2000','2001','2002','2003','2004']
Table_3 = summary_col([reg1, reg2, Val_2000_lm, Val_2001_lm, Val_2002_lm, Val_2003_lm, Val_2004_lm,],\
                      regressor_order=['OBP','SLG','PA','arb','free_agent','catcher','Infld','Intercept'],stars=True, \
                      float_format="'%.3f'",model_names = Header)
print(Table_3)


                   PriorMoneyball  Moneyball      2000        2001        2002        2003        2004   
---------------------------------------------------------------------------------------------------------
OBP                '1.447'**      '5.400'***  '2.986'**   '0.683'     '0.731'     '3.012'*    '7.559'*** 
                   ('0.698')      ('1.079')   ('1.463')   ('1.461')   ('1.827')   ('1.819')   ('1.852')  
SLG                '2.961'***     '2.932'***  '2.819'***  '3.395'***  '2.347'***  '2.197'**   '3.056'*** 
                   ('0.312')      ('0.473')   ('0.667')   ('0.649')   ('0.763')   ('0.869')   ('0.847')  
PA                 '0.003'***     '0.003'***  '0.002'***  '0.003'***  '0.003'***  '0.003'***  '0.003'*** 
                   ('0.000')      ('0.000')   ('0.000')   ('0.000')   ('0.000')   ('0.000')   ('0.000')  
catcher            '0.126'**      '0.095'     '0.048'     '0.167'     '0.075'     '0.282'**   '0.086'    
                   ('0.053')      ('0.078')  

In [741]:
Master.columns.to_list()

['playerID',
 'yearID',
 'stint',
 'G',
 'AB',
 'R',
 'H',
 'Doubles',
 'Triples',
 'HR',
 'RBI',
 'SB',
 'CS',
 'BB',
 'SO',
 'IBB',
 'HBP',
 'SH',
 'SF',
 'GIDP',
 'OBP',
 'TB',
 'SLG',
 'PA',
 'SalYear',
 'salary',
 'teamID',
 'debut',
 'debut_year',
 'years_xp',
 'arb',
 'free_agent',
 'G_all',
 'GS',
 'G_batting',
 'G_defense',
 'G_p',
 'G_c',
 'G_1b',
 'G_2b',
 'G_3b',
 'G_ss',
 'G_lf',
 'G_cf',
 'G_rf',
 'G_of',
 'G_dh',
 'G_ph',
 'G_pr',
 'max_g',
 'pos',
 'catcher',
 '2B',
 'SS',
 '3B',
 'Infld',
 'lnSal',
 'BA']